0. IMPORTS

In [1]:
import os
import time
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from datasets import Dataset
from datetime import timedelta
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback, 
    TrainerCallback
)

1. PATHS & CONFIGURATION

In [2]:
MODEL_NAME = "ai4bharat/IndicBERTv2-MLM-only"
TRAIN_PATH = r'D:\Major Project\SpamX\machine_learning\dataset\train.csv'
VAL_PATH = r'D:\Major Project\SpamX\machine_learning\dataset\validation.csv'
OUTPUT_DIR = r'D:\Major Project\SpamX\machine_learning\models\IndicBERT'

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

c:\Users\jeeve\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


2. ETA & PROGRESS LOGGING

In [4]:
class ETALoggingCallback(TrainerCallback):
    def __init__(self):
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        print(f"\nIndicBERT v2 Finetuning Started | Total Steps: {state.max_steps}")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if self.start_time and state.global_step > 0:
            elapsed = time.time() - self.start_time
            avg_time_per_step = elapsed / state.global_step
            remaining_steps = state.max_steps - state.global_step
            eta_seconds = remaining_steps * avg_time_per_step
            
            eta_str = str(timedelta(seconds=int(eta_seconds)))
            raw_loss = logs.get("loss", "N/A")
            loss_val = f"{raw_loss:.4f}" if isinstance(raw_loss, (float, int)) else "Calculating..."
            
            print(f"Step: {state.global_step}/{state.max_steps} | Loss: {loss_val} | ETA: {eta_str} | Progress: {(state.global_step/state.max_steps)*100:.2f}%")

3. SLIDING WINDOW

In [5]:
def tokenize_comment_only(examples):
    texts = [str(com) for com in examples["Comment"]]
    
    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=128, 
        stride=32,      
        return_overflowing_tokens=True,
        padding="max_length"
    )
    
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    tokenized["labels"] = [examples["Label"][i] for i in sample_mapping]
    return tokenized

4. METRICS

In [6]:
class FocalLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        gamma, alpha = 2.0, 0.25 
        probs = F.softmax(logits, dim=-1)
        pt = probs.gather(1, labels.unsqueeze(1)).squeeze(1)
        
        loss = -alpha * (1 - pt) ** gamma * (pt + 1e-8).log()
        return (loss.mean(), outputs) if return_outputs else loss.mean()

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
        "precision": precision_score(labels, predictions),
        "recall": recall_score(labels, predictions)
    }

5. DATA LOADING & PREPARATION

In [7]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

train_ds = Dataset.from_pandas(train_df).map(tokenize_comment_only, batched=True, remove_columns=train_df.columns.tolist())
val_ds = Dataset.from_pandas(val_df).map(tokenize_comment_only, batched=True, remove_columns=val_df.columns.tolist())

Map:   0%|          | 0/61645 [00:00<?, ? examples/s]

Map:   0%|          | 0/6438 [00:00<?, ? examples/s]

6. MODEL INITIALIZATION & FINETUNING

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, trust_remote_code=True
).to(device)

training_args = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "checkpoints"),
    num_train_epochs=3,
    per_device_train_batch_size=16, 
    learning_rate=2e-5,
    logging_steps=50,
    evaluation_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True, 
    report_to="none",
    disable_tqdm=True
)

trainer = FocalLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2), ETALoggingCallback()]
)

trainer.train(resume_from_checkpoint=True)

c:\Users\jeeve\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ai4bharat/IndicBERTv2-MLM-only and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\jeeve\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\trainer.py:2956: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pyt


IndicBERT v2 Finetuning Started | Total Steps: 11676


c:\Users\jeeve\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\trainer.py:2699: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint_rng_state = t

Step: 3050/11676 | Loss: 0.0087 | ETA: 0:01:25 | Progress: 26.12%
{'loss': 0.0087, 'grad_norm': 0.6291441917419434, 'learning_rate': 1.4777321000342583e-05, 'epoch': 0.7836587872559095}
Step: 3100/11676 | Loss: 0.0069 | ETA: 0:02:40 | Progress: 26.55%
{'loss': 0.0069, 'grad_norm': 0.12967173755168915, 'learning_rate': 1.4691675231243578e-05, 'epoch': 0.7965056526207606}
Step: 3150/11676 | Loss: 0.0070 | ETA: 0:03:51 | Progress: 26.98%
{'loss': 0.007, 'grad_norm': 0.2428465187549591, 'learning_rate': 1.460602946214457e-05, 'epoch': 0.8093525179856115}
Step: 3200/11676 | Loss: 0.0076 | ETA: 0:05:00 | Progress: 27.41%
{'loss': 0.0076, 'grad_norm': 0.4705566465854645, 'learning_rate': 1.4520383693045565e-05, 'epoch': 0.8221993833504625}
Step: 3250/11676 | Loss: 0.0061 | ETA: 0:06:05 | Progress: 27.83%
{'loss': 0.0061, 'grad_norm': 0.5845422744750977, 'learning_rate': 1.4434737923946558e-05, 'epoch': 0.8350462487153134}
Step: 3300/11676 | Loss: 0.0069 | ETA: 0:07:08 | Progress: 28.26%
{'los

TrainOutput(global_step=7500, training_loss=0.0027424385239680606, metrics={'train_runtime': 2771.8135, 'train_samples_per_second': 67.39, 'train_steps_per_second': 4.212, 'train_loss': 0.0027424385239680606, 'epoch': 1.9270298047276464})

7. FINAL SAVE

In [9]:
final_path = os.path.join(OUTPUT_DIR, "final_indicbert")
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)

print(f"\nIndicBERT v2 finetuning complete! Saved to: {final_path}")


IndicBERT v2 finetuning complete! Saved to: D:\Major Project\SpamX\machine_learning\models\IndicBERT\final_indicbert
